In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import numpy as np
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

FIGURES = Path('../outputs/figures')
FIGURES.mkdir(parents=True, exist_ok=True)

df = pd.read_csv('../Data:/grade_level_gaps.csv')
df['grade'] = df['grade'].astype(int)
print(df.shape)
df.head(3)

## Figure 1 — Govt–Private Arithmetic Gap by Grade and Province

The gap is defined as the difference in mean arithmetic scores between private and government school students at the same grade level (positive = private advantage). Trajectories are shown for the national aggregate and four major provinces — Punjab, Sindh, KPK, and Balochistan — covering grades 1 through 10.

In [ ]:
PROVINCES = ['National', 'Punjab', 'Sindh', 'KPK', 'Balochistan']
PALETTE = {
    'National':    '#2c2c2c',
    'Punjab':      '#1f77b4',
    'Sindh':       '#d62728',
    'KPK':         '#2ca02c',
    'Balochistan': '#ff7f0e',
}
STYLES = {
    'National':    (None, 2.5),
    'Punjab':      ((6, 2), 1.8),
    'Sindh':       ((2, 2), 1.8),
    'KPK':         ((6, 2, 2, 2), 1.8),
    'Balochistan': ((1, 1), 1.8),
}

fig, ax = plt.subplots(figsize=(9, 5))

for prov in PROVINCES:
    sub = df[df['province'] == prov].sort_values('grade')
    dash, lw = STYLES[prov]
    ax.plot(
        sub['grade'], sub['arithmetic_gap'],
        color=PALETTE[prov], linewidth=lw,
        dashes=dash if dash else (None, None),
        marker='o', markersize=4.5, markerfacecolor='white',
        markeredgewidth=1.4, label=prov,
    )

ax.axhline(0, color='#999999', linewidth=0.8, linestyle='--', zorder=0)
ax.set_xlabel('Grade', labelpad=8)
ax.set_ylabel('Arithmetic gap\n(private − government)', labelpad=8)
ax.set_title('Govt–Private Arithmetic Gap by Grade', fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(range(1, 11))
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
ax.legend(frameon=False, loc='upper right', fontsize=10)
sns.despine(ax=ax)
plt.tight_layout()

fig.savefig(FIGURES / '01_arithmetic_gap_by_grade_province.png', bbox_inches='tight')
fig.savefig(FIGURES / '01_arithmetic_gap_by_grade_province.pdf', bbox_inches='tight')
plt.show()

## Figure 2 — Government vs Private Mean Arithmetic Score at Key Grades (National)

Grouped bars compare the absolute mean arithmetic score for government and private school students at benchmark grades (3, 5, 8, 10) using the national aggregate. This illustrates how both sectors improve with grade while a consistent private advantage persists.

In [ ]:
KEY_GRADES = [3, 5, 8, 10]
nat = df[df['province'] == 'National'].set_index('grade')
sub = nat.loc[KEY_GRADES]

x = np.arange(len(KEY_GRADES))
w = 0.38

fig, ax = plt.subplots(figsize=(8, 5))

bars_g = ax.bar(x - w / 2, sub['govt_mean_arithmetic'], width=w,
                label='Government', color='#4c72b0', zorder=3)
bars_p = ax.bar(x + w / 2, sub['pvt_mean_arithmetic'], width=w,
                label='Private', color='#dd8452', zorder=3)

for bar in list(bars_g) + list(bars_p):
    h = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2, h + 0.05,
        f'{h:.2f}', ha='center', va='bottom', fontsize=9, color='#444444'
    )

ax.set_xlabel('Grade', labelpad=8)
ax.set_ylabel('Mean arithmetic score', labelpad=8)
ax.set_title('Government vs Private Mean Arithmetic Score — National', fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(x)
ax.set_xticklabels([f'Grade {g}' for g in KEY_GRADES])
ax.set_ylim(0, sub[['govt_mean_arithmetic', 'pvt_mean_arithmetic']].values.max() * 1.18)
ax.legend(frameon=False, fontsize=10)
ax.grid(axis='x', visible=False)
sns.despine(ax=ax)
plt.tight_layout()

fig.savefig(FIGURES / '02_govt_vs_pvt_arithmetic_key_grades.png', bbox_inches='tight')
fig.savefig(FIGURES / '02_govt_vs_pvt_arithmetic_key_grades.pdf', bbox_inches='tight')
plt.show()

## Figure 3 — Govt–Private Arithmetic Gap by Grade and Gender

The gender-disaggregated gap trajectories reveal whether the private school advantage accrues differently to male and female students as they progress through grades 1–10. Gender slices are stored in the same `province` field as `Gender: Male` and `Gender: Female`.

In [ ]:
GENDER_MAP = {'Gender: Male': 'Male', 'Gender: Female': 'Female'}
GENDER_PALETTE = {'Gender: Male': '#3a86ff', 'Gender: Female': '#ff006e'}
GENDER_MARKERS = {'Gender: Male': 'o', 'Gender: Female': 's'}

fig, ax = plt.subplots(figsize=(9, 5))

for key, label in GENDER_MAP.items():
    sub = df[df['province'] == key].sort_values('grade')
    ax.plot(
        sub['grade'], sub['arithmetic_gap'],
        color=GENDER_PALETTE[key], linewidth=2,
        marker=GENDER_MARKERS[key], markersize=5,
        markerfacecolor='white', markeredgewidth=1.5,
        label=label,
    )

ax.axhline(0, color='#999999', linewidth=0.8, linestyle='--', zorder=0)
ax.set_xlabel('Grade', labelpad=8)
ax.set_ylabel('Arithmetic gap\n(private − government)', labelpad=8)
ax.set_title('Govt–Private Arithmetic Gap by Grade and Gender', fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(range(1, 11))
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
ax.legend(frameon=False, fontsize=10)
sns.despine(ax=ax)
plt.tight_layout()

fig.savefig(FIGURES / '03_arithmetic_gap_by_grade_gender.png', bbox_inches='tight')
fig.savefig(FIGURES / '03_arithmetic_gap_by_grade_gender.pdf', bbox_inches='tight')
plt.show()